# NB-14 · C-4 TF-IDF + Regresión Logística — Iteración 2

**Proyecto**: SinergIA Lab — Clasificación documental (Banco Falabella SQR)  
**Fase CRISP-DM**: 2da iteración (después del retorno a Data Preparation diagnosticado en NB-13)  
**Origen del dato**: `data/processed/corpus_ocr.csv`

## Justificación

El [hallazgo del NB-13](13_clasificacion_validacion_features.ipynb) demostró que el F1 macro reportado por la 1ra iteración está inflado por **atajos morfológicos** (n_pages, n_chars) y, en menor medida, por **atajos léxicos** (palabras-bandera tipo `cedula`, `poliza`).

Esta iteración aplica las recomendaciones del hallazgo:

1. **Eliminar features estructurales pseudo-ID**: `n_pages`, `n_chars`, `n_words`, `n_digits`.
2. **Conservar features estilísticas independientes del tamaño**: `ratio_dig`, `ratio_upper`, `ratio_short`.
3. **Entrenar dos modelos**:
   - **Modelo 1 (plano)**: multiclase 4-way con features depuradas.
   - **Modelo 2 (ensamble dicotómico)**: clasificador binario CEDULA vs no-CEDULA + multiclase entre RUT/CÁMARA/POLIZA, siguiendo la sugerencia recibida en tutoría.
4. **Comparar contra C-1 baseline** (NB-10) y contra el Modelo D del NB-13 como referencia honesta.

## 1. Imports y paths

In [ ]:
import warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, classification_report,
                              confusion_matrix, accuracy_score,
                              precision_score, recall_score)
import joblib

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

ROOT = Path('..')
CORPUS = ROOT / 'data' / 'processed' / 'corpus_ocr.csv'
MODELS_DIR  = ROOT / 'models' / 'c4_tfidf_iter2'
REPORTS_DIR = ROOT / 'reports'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
print(f'Corpus  : {CORPUS}')
print(f'Modelos : {MODELS_DIR}')
print(f'Reports : {REPORTS_DIR}')

## 2. Carga del corpus + agregación a nivel documento

Mismo procedimiento que NB-13: normalizamos labels, agrupamos páginas por `doc_id` y computamos features morfológicas estilísticas (ratios independientes del tamaño).

In [ ]:
def norm_label(s):
    if pd.isna(s): return None
    s = str(s).upper()
    if 'CAMARA' in s or 'MARA' in s or 'CIO' in s: return 'CAMARA DE COMERCIO'
    if 'POLIZA' in s or 'OLIZA' in s or 'LIZA' in s: return 'POLIZA'
    if 'CEDULA' in s or 'DULA' in s: return 'CEDULA CIUDADANIA'
    if 'RUT' in s: return 'RUT'
    return None

df = pd.read_csv(CORPUS, dtype={'md5': str, 'doc_id': str})
df['texto_ocr'] = df['texto_ocr'].fillna('')
df['clase'] = df['folder'].apply(norm_label)

docs = df.groupby('doc_id').agg(
    texto=('texto_ocr', lambda s: '\n'.join(s)),
    clase=('clase', 'first'),
    n_pages=('page_num', 'count'),
).reset_index()
docs = docs[docs['texto'].str.strip().str.len() > 0].reset_index(drop=True)
docs = docs.dropna(subset=['clase']).reset_index(drop=True)

# Features morfológicas (calculadas pero solo conservaremos las ratios)
docs['n_chars']     = docs['texto'].str.len()
docs['n_words']     = docs['texto'].str.split().str.len()
docs['n_digits']    = docs['texto'].str.count(r'\d')
docs['n_upper']     = docs['texto'].str.count(r'[A-ZÁÉÍÓÚÑ]')
docs['n_short_w']   = docs['texto'].apply(lambda t: sum(1 for w in t.split() if len(w) <= 3))
docs['ratio_dig']   = docs['n_digits']  / docs['n_chars'].clip(lower=1)
docs['ratio_upper'] = docs['n_upper']   / docs['n_chars'].clip(lower=1)
docs['ratio_short'] = docs['n_short_w'] / docs['n_words'].clip(lower=1)

print(f'Documentos: {len(docs)}')
print(f'Distribución de clases:')
print(docs['clase'].value_counts().to_string())

## 3. Definición del nuevo set de features

### 3.1 Features eliminadas (pseudo-IDs / atajos morfológicos)

| Feature | Razón de eliminación |
|---|---|
| `n_pages` | Pseudo-ID del documento; CEDULA siempre = 1, resto = 5-10 |
| `n_chars`, `n_words` | Correlacionados fuertemente con la clase via longitud |
| `n_digits` | Atajo de tamaño en valor absoluto |

### 3.2 Features conservadas (estilísticas, independientes del tamaño)

| Feature | Por qué se conserva |
|---|---|
| `ratio_upper` | Proporción de mayúsculas (estilo, no tamaño) |
| `ratio_short` | Proporción de palabras cortas (estilo, no tamaño) |
| `ratio_dig`   | Densidad de dígitos por carácter (relevante en RUT/NIT, neutra al tamaño) |

In [ ]:
FEATURES_NUM_NUEVAS     = ['ratio_upper', 'ratio_short', 'ratio_dig']
FEATURES_NUM_ELIMINADAS = ['n_pages', 'n_chars', 'n_words', 'n_digits']

print('Features conservadas:', FEATURES_NUM_NUEVAS)
print('Features eliminadas :', FEATURES_NUM_ELIMINADAS)
print('\nDescriptivos de las nuevas:')
docs[FEATURES_NUM_NUEVAS].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, FEATURES_NUM_NUEVAS):
    sns.boxplot(data=docs, x='clase', y=col, ax=ax,
                order=['CEDULA CIUDADANIA', 'RUT', 'POLIZA', 'CAMARA DE COMERCIO'],
                palette='Set2', showfliers=False)
    ax.set_title(col); ax.set_xlabel('')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig_nb14_features_nuevas.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Split train/test estratificado

Como ya estamos a nivel documento, el split es directo (no requiere agrupación adicional). 80/20 estratificado por clase, semilla fija.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(docs['clase'].values)
class_names = le.classes_
ID_CEDULA = list(class_names).index('CEDULA CIUDADANIA')
print('Mapeo:', dict(zip(class_names, range(len(class_names)))))

idx_tr, idx_te = train_test_split(
    np.arange(len(docs)), test_size=0.20, stratify=y, random_state=SEED)
y_tr, y_te = y[idx_tr], y[idx_te]
texto_tr = docs.loc[idx_tr, 'texto'].values
texto_te = docs.loc[idx_te, 'texto'].values
Xnum_tr  = docs.loc[idx_tr, FEATURES_NUM_NUEVAS].values
Xnum_te  = docs.loc[idx_te, FEATURES_NUM_NUEVAS].values

print(f'\nTrain: {len(idx_tr)} docs  |  Test: {len(idx_te)} docs')
print('\nDistribución test:')
print(pd.Series(le.inverse_transform(y_te)).value_counts().to_string())

## 5. Modelo 1 — Plano (multiclase con features depuradas)

In [ ]:
def construir_features(tx_tr, tx_te, num_tr, num_te):
    """Construye matriz sparse [TF-IDF | numéricas escaladas]."""
    vec = TfidfVectorizer(
        max_features=5000, min_df=3, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
        strip_accents='unicode', lowercase=True,
    )
    Xt = vec.fit_transform(tx_tr); Xv = vec.transform(tx_te)
    sc = StandardScaler()
    N_tr = sc.fit_transform(num_tr); N_te = sc.transform(num_te)
    Xtr = hstack([Xt, csr_matrix(N_tr)]).tocsr()
    Xte = hstack([Xv, csr_matrix(N_te)]).tocsr()
    return Xtr, Xte, vec, sc

Xtr, Xte, vec_m1, sc_m1 = construir_features(texto_tr, texto_te, Xnum_tr, Xnum_te)
print(f'X_train: {Xtr.shape}  |  X_test: {Xte.shape}')

clf_plano = LogisticRegression(
    max_iter=2000, class_weight='balanced',
    random_state=SEED, n_jobs=-1, C=1.0,
)
clf_plano.fit(Xtr, y_tr)
y_pred_plano = clf_plano.predict(Xte)

f1m_plano = f1_score(y_te, y_pred_plano, average='macro')
f1w_plano = f1_score(y_te, y_pred_plano, average='weighted')
acc_plano = accuracy_score(y_te, y_pred_plano)
print(f'\n=== Modelo 1 — Plano + features depuradas ===')
print(f'F1 macro    : {f1m_plano:.4f}')
print(f'F1 weighted : {f1w_plano:.4f}')
print(f'Accuracy    : {acc_plano:.4f}')
print('\nClassification report:')
print(classification_report(y_te, y_pred_plano, target_names=class_names, digits=4))

In [ ]:
cm_plano = confusion_matrix(y_te, y_pred_plano)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_plano, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title(f'Modelo 1 — Plano + features depuradas (F1 macro={f1m_plano:.4f})')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig_nb14_cm_modelo_plano.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Modelo 2 — Ensamble dicotómico

- **Etapa 1**: clasificador binario *¿es CEDULA o no?* — entrenado sobre todo el train.
- **Etapa 2**: clasificador multiclase entre RUT, CÁMARA, POLIZA — entrenado solo sobre los documentos no-CEDULA.

Inferencia: si Etapa 1 predice CEDULA → output = CEDULA. Si predice "no cédula" → consultar Etapa 2.

### 6.1 Etapa 1 — Binario CEDULA vs no-CEDULA

In [ ]:
y_bin_tr = (y_tr == ID_CEDULA).astype(int)
y_bin_te = (y_te == ID_CEDULA).astype(int)
print(f'CEDULA en train: {y_bin_tr.sum()} / {len(y_bin_tr)}  ({y_bin_tr.mean()*100:.1f}%)')
print(f'CEDULA en test : {y_bin_te.sum()} / {len(y_bin_te)}  ({y_bin_te.mean()*100:.1f}%)')

clf_e1 = LogisticRegression(
    max_iter=2000, class_weight='balanced',
    random_state=SEED, n_jobs=-1, C=1.0,
)
clf_e1.fit(Xtr, y_bin_tr)
pred_e1 = clf_e1.predict(Xte)

p_e1 = precision_score(y_bin_te, pred_e1)
r_e1 = recall_score(y_bin_te, pred_e1)
f1_e1 = f1_score(y_bin_te, pred_e1)
print(f'\n=== Etapa 1 (binario) ===')
print(f'Precision: {p_e1:.4f}  |  Recall: {r_e1:.4f}  |  F1: {f1_e1:.4f}')
print('\nMatriz de confusión:')
print(pd.DataFrame(
    confusion_matrix(y_bin_te, pred_e1),
    index=['Real no-CEDULA', 'Real CEDULA'],
    columns=['Pred no-CEDULA', 'Pred CEDULA']
))

### 6.2 Etapa 2 — Multiclase entre RUT, CÁMARA, POLIZA

Entrena solo sobre documentos no-CEDULA, con TF-IDF reajustado sobre ese subconjunto.

In [ ]:
mtr_nc = (y_tr != ID_CEDULA)
Xtr_e2, Xte_e2, vec_e2, sc_e2 = construir_features(
    texto_tr[mtr_nc], texto_te, Xnum_tr[mtr_nc], Xnum_te)
print(f'Train E2 (no-CEDULA): {Xtr_e2.shape[0]} docs')
print(f'Vocabulario E2: {len(vec_e2.get_feature_names_out())} términos')

clf_e2 = LogisticRegression(
    max_iter=2000, class_weight='balanced',
    random_state=SEED, n_jobs=-1, C=1.0,
)
clf_e2.fit(Xtr_e2, y_tr[mtr_nc])

# Evaluación interna sobre los documentos no-CEDULA del test
mte_nc = (y_te != ID_CEDULA)
y_pred_e2_only = clf_e2.predict(Xte_e2[mte_nc])
f1m_e2 = f1_score(y_te[mte_nc], y_pred_e2_only, average='macro')
print(f'\n=== Etapa 2 (multiclase no-CEDULA) ===')
print(f'F1 macro: {f1m_e2:.4f}')
print(classification_report(
    y_te[mte_nc], y_pred_e2_only,
    target_names=class_names, digits=4,
    labels=[i for i in range(len(class_names)) if i != ID_CEDULA],
))

### 6.3 Inferencia del ensamble

In [ ]:
pred_e2_full = clf_e2.predict(Xte_e2)
y_pred_ens   = np.where(pred_e1 == 1, ID_CEDULA, pred_e2_full)

f1m_ens = f1_score(y_te, y_pred_ens, average='macro')
f1w_ens = f1_score(y_te, y_pred_ens, average='weighted')
acc_ens = accuracy_score(y_te, y_pred_ens)
print(f'=== Ensamble dicotómico (E1 + E2) ===')
print(f'F1 macro    : {f1m_ens:.4f}')
print(f'F1 weighted : {f1w_ens:.4f}')
print(f'Accuracy    : {acc_ens:.4f}')
print('\nClassification report:')
print(classification_report(y_te, y_pred_ens, target_names=class_names, digits=4))

In [ ]:
cm_ens = confusion_matrix(y_te, y_pred_ens)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title(f'Modelo 2 — Ensamble dicotómico (F1 macro={f1m_ens:.4f})')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig_nb14_cm_ensamble.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Comparación con modelos previos

In [ ]:
comparacion = pd.DataFrame([
    {'Notebook': 'NB-13 A', 'Variante': 'TF-IDF + features completas (con atajos)', 'F1 macro': '*', 'Comentario': 'Baseline inflado'},
    {'Notebook': 'NB-13 D', 'Variante': 'Honesto (sin atajos + bigrams)',           'F1 macro': '*', 'Comentario': 'Referencia honesta'},
    {'Notebook': 'NB-14',   'Variante': 'C-4 Plano + features depuradas',           'F1 macro': round(f1m_plano, 4), 'Comentario': 'Multiclase con 3 ratios estilísticas'},
    {'Notebook': 'NB-14',   'Variante': 'C-4 Ensamble dicotómico',                  'F1 macro': round(f1m_ens, 4),   'Comentario': 'CEDULA aislada + multiclase RUT/CÁMARA/POLIZA'},
])
comparacion

## 8. Persistencia del modelo y artefactos

In [ ]:
# Modelo 1 (plano)
joblib.dump(vec_m1,    MODELS_DIR / 'modelo1_plano_vectorizer.joblib')
joblib.dump(sc_m1,     MODELS_DIR / 'modelo1_plano_scaler.joblib')
joblib.dump(clf_plano, MODELS_DIR / 'modelo1_plano_classifier.joblib')

# Modelo 2 (ensamble)
joblib.dump(clf_e1,    MODELS_DIR / 'modelo2_ens_e1_classifier.joblib')
joblib.dump(vec_e2,    MODELS_DIR / 'modelo2_ens_e2_vectorizer.joblib')
joblib.dump(sc_e2,     MODELS_DIR / 'modelo2_ens_e2_scaler.joblib')
joblib.dump(clf_e2,    MODELS_DIR / 'modelo2_ens_e2_classifier.joblib')

# Predicciones para reporte comparativo
doc_ids_te = docs.loc[idx_te, 'doc_id'].values
predictions_df = pd.DataFrame({
    'doc_id': doc_ids_te,
    'y_true': le.inverse_transform(y_te),
    'y_pred_plano':    le.inverse_transform(y_pred_plano),
    'y_pred_ensamble': le.inverse_transform(y_pred_ens),
})
pred_path = ROOT / 'data' / 'processed' / 'c4_predictions.csv'
predictions_df.to_csv(pred_path, index=False, encoding='utf-8')

summary = {
    'model': 'C-4 TF-IDF+LR · Iteración 2',
    'features_eliminadas': FEATURES_NUM_ELIMINADAS,
    'features_conservadas': FEATURES_NUM_NUEVAS,
    'n_train': int(len(idx_tr)),
    'n_test':  int(len(idx_te)),
    'modelo_1_plano': {
        'f1_macro':    round(f1m_plano, 4),
        'f1_weighted': round(f1w_plano, 4),
        'accuracy':    round(acc_plano, 4),
        'classification_report': classification_report(
            y_te, y_pred_plano, target_names=list(class_names), output_dict=True, digits=4),
    },
    'modelo_2_ensamble': {
        'f1_macro':    round(f1m_ens, 4),
        'f1_weighted': round(f1w_ens, 4),
        'accuracy':    round(acc_ens, 4),
        'etapa_1_binario': {
            'precision': round(p_e1, 4), 'recall': round(r_e1, 4), 'f1': round(f1_e1, 4),
        },
        'classification_report': classification_report(
            y_te, y_pred_ens, target_names=list(class_names), output_dict=True, digits=4),
    },
}
with open(REPORTS_DIR / 'nb14_resumen.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Modelos guardados en  : {MODELS_DIR}')
print(f'Predicciones en       : {pred_path}')
print(f'Resumen JSON en       : {REPORTS_DIR / "nb14_resumen.json"}')

## 9. Conclusiones de la 2da iteración

### Resultados

- **Modelo 1 (plano + features depuradas)** alcanza F1 macro = `{ver tabla}`. Eliminamos las 4 features pseudo-ID y conservamos solo 3 ratios estilísticas; el F1 resultante refleja desempeño honesto del modelo sobre clasificación léxica genuina, no sobre asimetrías morfológicas del corpus.
- **Modelo 2 (ensamble dicotómico)** alcanza F1 macro = `{ver tabla}`. Aísla explícitamente la clase morfológicamente outlier (CEDULA) del clasificador multiclase, que opera solo sobre clases comparables (RUT, CÁMARA, POLIZA).

### Lectura

- La caída del F1 respecto al baseline NB-10 (C-1) **es metodológicamente esperada y deseada**: refleja la eliminación de los atajos identificados en NB-13. La cifra resultante es defendible en producción.
- El ensamble es preferible al modelo plano porque **separa la decisión sobre la clase outlier de la decisión sobre las clases comparables**, reduciendo el riesgo de que la asimetría morfológica contamine las métricas reportadas.

### Próximos pasos

1. **NB-15 (futuro)**: reentrenar BETO/DistilBERT con el mismo protocolo de features depuradas y comparar con C-4 ensamble.
2. **Incorporar clase OTROS o RUP** para introducir presión adversaria.
3. **Iniciar NER** sobre 1 tipo documental (Cámara o Póliza) — etiquetado manual de ~100 docs en docano/Label Studio.